# 🎬 FFmpeg AI Video Editor Web UI trên Google Colab

Notebook này giúp bạn chạy toàn bộ Web UI dựng video trên **Google Colab**:
- ⚡ **Không tốn CPU/RAM máy tính:** Toàn bộ quá trình FFmpeg render video, hiệu ứng, phụ đề chạy trên đám mây của Google.
- 🚀 **Không lo lỗi Timeout 524:** Sử dụng Localtunnel / Pinggy không giới hạn thời gian render video dài.
- 🌐 **Truy cập trực tiếp trên trình duyệt máy tính.**

---

### 📌 BƯỚC 1: Cài đặt Node.js 20, FFmpeg & Công cụ đường truyền Tunnel
*(Chạy ô này một lần khi mới mở Colab)*

In [ ]:
# 1. Cập nhật hệ thống và cài đặt FFmpeg
!apt-get update -qq && apt-get install -y -qq ffmpeg

# 2. Cài đặt Node.js v20 LTS
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y -qq nodejs

# 3. Cài đặt localtunnel toàn cục (tránh lỗi Timeout 524 khi render video lâu)
!npm install -g localtunnel > /dev/null 2>&1

print("✅ Cài đặt môi trường thành công!")
!node -v
!npm -v
!ffmpeg -version | head -n 1

### 📌 BƯỚC 2: Kết nối Google Drive
*(Để lưu video render và dữ liệu vĩnh viễn)*

In [ ]:
import os
from google.colab import drive

# Kết nối Google Drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/VideoTool_Edit'

if not os.path.exists(PROJECT_DIR):
    os.makedirs(PROJECT_DIR, exist_ok=True)

%cd {PROJECT_DIR}
print(f"📂 Đang làm việc tại: {os.getcwd()}")

### 📌 BƯỚC 3: Giải nén project.zip (Nếu có file mới) & Cài đặt thư viện

In [ ]:
import os

# Nếu có upload file project.zip ở thư mục gốc Colab, giải nén vào Drive:
if os.path.exists('/content/project.zip'):
    print("📦 Đang giải nén project.zip vào Google Drive...")
    !unzip -q -o /content/project.zip -d .

# Cài đặt dependencies
!npm install
print("✅ Cài đặt thư viện dự án hoàn tất!")

### 📌 BƯỚC 4: 🚀 KHỞI ĐỘNG WEB UI (Không bao giờ bị lỗi Timeout 524)

Chạy ô bên dưới để lấy link mở Web UI:

In [ ]:
import subprocess
import time
import urllib.request

# 1. Khởi động Node.js server
print("🚀 Đang khởi động Server Node.js...")
server_proc = subprocess.Popen(["node", "server.js"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(3)

# 2. Lấy IP máy ảo Colab để mở mật khẩu Localtunnel nếu cần
try:
    public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
except:
    public_ip = ""

# 3. Khởi động Localtunnel (Không giới hạn timeout 100s)
print("🌐 Đang tạo đường hầm Localtunnel...")
lt_proc = subprocess.Popen(["npx", "localtunnel", "--port", "3000"], stdout=subprocess.PIPE, text=True)

time.sleep(4)
lt_url = ""
for _ in range(5):
    line = lt_proc.stdout.readline()
    if "url is:" in line:
        lt_url = line.split("url is:")[1].strip()
        break
    time.sleep(1)

print("\n" + "="*65)
if lt_url:
    print(f"🎉 LINK WEB UI CHÍNH: {lt_url}")
    if public_ip:
        print(f"🔑 Nếu trang web hỏi Password/Endpoint IP, bạn nhập IP này: {public_ip}")
else:
    print("⚠️ Đang tạo link dự phòng qua Pinggy...")
    !ssh -o StrictHostKeyChecking=no -R0:localhost:3000 a.pinggy.io
print("="*65 + "\n")

# Giữ server chạy liên tục
try:
    while True:
        output = server_proc.stdout.readline()
        if output:
            print(output.strip())
        time.sleep(0.5)
except KeyboardInterrupt:
    print("🛑 Đã dừng server.")
    server_proc.terminate()
    lt_proc.terminate()